In [ ]:
!pip install numpy scipy

In [ ]:
import numpy as np
import scipy as sp
import random
from numpy import pi
import matplotlib.pyplot as plt
%matplotlib inline

# Вариант 6

### Задание

1. Ознакомиться с теоретической частью.

2. На основе лабораторной работы 1, реализовать однородный, КИХ и БИХ фильтр:

    - **однородный**: рекурсивный M=159;
  
    - **КИХ**: полосовой, Ханна, 200-800 Гц M=151;
  
    - **БИХ**: однополюсный НЧ fc=350 Гц;
  
    ---
    
    - рассчитать необходимые для реализации фильтра коэффициенты;
    
    - изменить сигнал из лабораторной работы 1 так, чтобы экспериментально подтвердить правильность работы соответствующего фильтра;
  
    - построить графики;
  
    - сохранить исходный, шумный и фильтрованый сигналы в WAV-файл.

#### Графики
  
- [x] входной функции;

- [ ] АЧХ спроектированных фильтров;

- [x] сигнала после внесения искажений;

- [ ] результаты работы однородного фильтра;

- [ ] результаты работы КИХ фильтра;

- [ ] результаты работы БИХ фильтра.

In [ ]:
A_x = [1, 0.5, 0.3, 0.1]
f0_x = 262 #Hz
h_x = [1, 2, 4, 8]
phi_x = 0

In [ ]:
def plot(y, x, title: str|None = None):
    plt.figure(figsize=(4, 3))
    plt.plot(x, y)
    plt.title(title)
    plt.grid(True)
    plt.show() 

In [ ]:
F = f0_x # base freq
F_S = 2 * 2*F*max(h_x) # optimal sampling freq
SAMPLE_COUNT = 1024

NOISE_HARMONICS = [
    (random.uniform(0.3, 0.9), random.uniform(max(h_x), 2*max(h_x))) # (amplitude, frequency multiplier)
    for i in range(3) # how many
]

In [ ]:
def s(t, A, h, f0, phi):
    assert(len(A) == len(h))
    return sum(
        A[i] * np.sin(2*pi * h[i] * f0 * t + phi)
        for i in range(0, len(A))
    )
x = lambda t: s(t, A_x, h_x, f0_x, phi_x)
x_noisy = lambda t: x(t) + sum([a * np.sin(2*pi * f_mul*F * t) for a, f_mul in NOISE_HARMONICS])

t = np.linspace(0, 5 / F, SAMPLE_COUNT)

plot(x(t), t, "x(t)")
plot(x_noisy(t), t, "x'(t)")

In [ ]:
t = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
f = np.linspace(0, F_S, SAMPLE_COUNT)

f = np.fft.fftshift(f) - F_S/2


np_fft_x = lambda t: np.fft.fft(x(t))
plot(np.abs(np_fft_x(t)), f, "NumPy FFT[x(t)] (amplitude domain)")
# plot(np.angle(np_fft_x(t)), f, "NumPy FFT[x(t)] (phase domain)")

np_fft_x_noisy = lambda t: np.fft.fft(x_noisy(t))
plot(np.abs(np_fft_x_noisy(t)), f, "NumPy FFT[x'(t)] (amplitude domain)")
# plot(np.angle(np_fft_x_noisy(t)), f, "NumPy FFT[x'(t)] (phase domain)")

In [ ]:
hanning_filter = lambda n: 0.5 * (1 - np.cos(2*pi*n/(len(n)-1)))

N = 60
n = np.linspace(0, N-1, N)
plot(hanning_filter(n), n)

---
# LEGACY vvvvvvvvvvvvvvvvvvvvvvv

### С использованием библиотек

In [ ]:
t = np.linspace(0, 5 / F, SAMPLE_COUNT)
t_ = np.linspace(0, 10 / F, 2*SAMPLE_COUNT-1)

np_conv = lambda t: np.convolve(x(t), y(t))
plot(np_conv(t), t_, "NumPy Convolution")

np_corr = lambda t: np.correlate(x(t), y(t), "full")
plot(np_corr(t), t_, "NumPy Correlation")

In [ ]:
def convolve(x, y, *, do_flip: bool = True):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([np.zeros(K-len(x)), x])
    y = np.concatenate([np.flip(y) if do_flip else y, np.zeros(K-len(y))])

    result = np.zeros(K)
    for i in range(len(result)):
        result[i] = sum(x*y)
        y = np.roll(y, 1)
    return result

def fast_convolve(x, y):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([x, np.zeros(K-len(x) + 1)])
    y = np.concatenate([y, np.zeros(K-len(y) + 1)])

    return np.real(inverse_fast_fourier_transform(fast_fourier_transform(x) * fast_fourier_transform(y))[:-1])
    
def correlate(x, y): return convolve(x, y, do_flip=False)
    
def fast_correlate(x, y):
    N = len(x)
    M = len(y)
    K = N+M-1
    
    x = np.concatenate([x, np.zeros(K-len(x) + 1)])
    y = np.concatenate([np.zeros(K-len(y) + 1), y])

    return np.real(inverse_fast_fourier_transform(fast_fourier_transform(x) * np.conj(fast_fourier_transform(y)))[:-1])
    
    x = np.concatenate([np.zeros(K-len(x)), x])
    y = np.concatenate([y, np.zeros(K-len(y))])

t = np.linspace(0, 5 / F, SAMPLE_COUNT)
t_ = np.linspace(0, 10 / F, 2*SAMPLE_COUNT-1)

conv = lambda t: convolve(x(t), y(t))
conv_fft = lambda t: fast_convolve(x(t), y(t))
plot(conv(t), t_, "conv(x(t), y(t))")
plot(conv_fft(t), t_, "conv(x(t), y(t)) (through FFT)")

corr = lambda t: correlate(x(t), y(t))
corr_fft = lambda t: fast_correlate(x(t), y(t))
plot(corr(t), t_, "corr(x(t), y(t))")
plot(corr_fft(t), t_, "corr(x(t), y(t)) (through FFT)")

In [ ]:
t = np.linspace(0, SAMPLE_COUNT / F_S, SAMPLE_COUNT)
f_ = np.linspace(0, 2*F_S, 2*SAMPLE_COUNT-1)

fft_conv = lambda t: fast_fourier_transform(np.concatenate([
    fast_convolve(x(t), y(t)), [0]
]))[:-1]
plot(np.abs(fft_conv(t)), f_, "FFT[conv(x(t), y(t))]")